In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv

In [ ]:
N_CLUSTERS = 5
K_NEIGHBORS = 8

EPOCHS = 200
LR = 0.005
WEIGHT_DECAY = 1e-4

GAT_HEADS = 4
HIDDEN = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
X_df = pd.read_parquet("../data/processed_predictors/atlanta_2023_predictor_dataset.parquet")

y_df = pd.read_parquet("../data/processed_job_accessibility/accessibility_state-GA_counties-121-089-067-135_year-2023_thresholds-15-30-45-60.parquet")

In [ ]:
y_df = y_df.sort_values("from_id").reset_index(drop=True)

weights = np.array([1.0, 0.7, 0.4, 0.2])

band_0_15 = y_df["jobs_15min"].clip(lower=0)
band_15_30 = (y_df["jobs_30min"] - y_df["jobs_15min"]).clip(lower=0)
band_30_45 = (y_df["jobs_45min"] - y_df["jobs_30min"]).clip(lower=0)
band_45_60 = (y_df["jobs_60min"] - y_df["jobs_45min"]).clip(lower=0)

weighted_sum = (
    weights[0] * band_0_15 +
    weights[1] * band_15_30 +
    weights[2] * band_30_45 +
    weights[3] * band_45_60
)

y_df["access_index"] = np.log1p(weighted_sum)

In [ ]:
df = X_df.merge(
    y_df[["from_id", "access_index"]],
    left_on="tract_id",
    right_on="from_id",
    how="inner"
)

In [ ]:
url = (
    "https://www2.census.gov/geo/tiger/TIGER2023/"
    "TRACT/tl_2023_13_tract.zip"
)

tracts = gpd.read_file(url)
tracts = tracts[tracts["COUNTYFP"].isin(["121", "089", "067", "135"])].copy()
tracts["tract_id"] = tracts["GEOID"].astype(str)

df = df.merge(tracts[["tract_id", "geometry"]], on="tract_id", how="left")

gdf = gpd.GeoDataFrame(df, geometry="geometry", crs=tracts.crs)
gdf = gdf.to_crs(epsg=5070)

In [ ]:
gdf["centroid"] = gdf.geometry.centroid
gdf["x"] = gdf.centroid.x
gdf["y"] = gdf.centroid.y

drop_cols = [
    "tract_id","from_id","geometry","centroid",
    "x","y","cluster","access_index"
]

feature_cols = [c for c in gdf.columns if c not in drop_cols]

X_np = gdf[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0).values

scaler = StandardScaler()
X_np = scaler.fit_transform(X_np)

X = torch.tensor(X_np, dtype=torch.float)
y = torch.tensor(gdf["access_index"].values, dtype=torch.float)

coords = np.column_stack([gdf["x"], gdf["y"]])

In [ ]:
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init="auto")
gdf["cluster"] = kmeans.fit_predict(coords)

In [ ]:
def build_knn_graph(indices, coords, k=K_NEIGHBORS):
    nn = NearestNeighbors(n_neighbors=k)
    nn.fit(coords[indices])

    _, idx = nn.kneighbors(coords[indices])

    edges = []
    for i_local, i_global in enumerate(indices):
        for j_local in idx[i_local]:
            j_global = indices[j_local]
            if i_global != j_global:
                edges.append((i_local, j_local))

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    return edge_index

In [ ]:
class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden=64, heads=4):
        super().__init__()

        self.gat1 = GATConv(in_channels, hidden, heads=heads, concat=True)
        self.gat2 = GATConv(hidden * heads, hidden, heads=1, concat=True)

        self.lin = torch.nn.Linear(hidden, 1)

    def forward(self, x, edge_index):
        x1 = F.elu(self.gat1(x, edge_index))
        x2 = F.elu(self.gat2(x1, edge_index))

        x = x1 + x2[:, :x1.shape[1]]  # residual alignment
        return self.lin(x).squeeze()

In [ ]:
results = []
all_predictions = []

for test_cluster in range(N_CLUSTERS):

    print("=" * 60)
    print(f"TEST CLUSTER: {test_cluster}")
    print("=" * 60)

    train_idx = np.where(gdf["cluster"] != test_cluster)[0]
    test_idx = np.where(gdf["cluster"] == test_cluster)[0]

    train_mask = torch.zeros(len(gdf), dtype=torch.bool, device=device)
    test_mask = torch.zeros(len(gdf), dtype=torch.bool, device=device)

    train_mask[train_idx] = True
    test_mask[test_idx] = True

    edge_index = build_knn_graph(train_idx, coords).to(device)

    model = GAT(in_channels=X.shape[1], hidden=HIDDEN, heads=GAT_HEADS).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = torch.nn.MSELoss()

    # ---------------------
    # TRAIN
    # ---------------------
    model.train()

    for epoch in range(EPOCHS):
        optimizer.zero_grad()

        out = model(X.to(device), edge_index)

        loss = loss_fn(out[train_mask], y.to(device)[train_mask])

        loss.backward()
        optimizer.step()

    # ---------------------
    # EVAL
    # ---------------------
    model.eval()

    with torch.no_grad():
        pred = model(X.to(device), edge_index).cpu().numpy()

    y_true = y[test_mask].cpu().numpy()
    y_pred = pred[test_mask.cpu().numpy()]

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2: {r2:.4f}")

    results.append({
        "cluster": test_cluster,
        "mae": mae,
        "rmse": rmse,
        "r2": r2
    })

    # ---------------------
    # STORE PREDICTIONS
    # ---------------------
    all_predictions.append(pd.DataFrame({
        "tract_id": gdf.loc[test_idx, "tract_id"].values,
        "actual": y_true,
        "predicted": y_pred[test_mask.cpu().numpy()]
    }))

TEST CLUSTER: 0


RuntimeError: The size of tensor a (256) must match the size of tensor b (64) at non-singleton dimension 1

In [ ]:
results_df = pd.DataFrame(results)

print(results_df)
print(results_df[["mae","rmse","r2"]].mean())

In [ ]:
pred_df = pd.concat(all_predictions)

plt.figure(figsize=(6,6))
plt.scatter(pred_df["actual"], pred_df["predicted"], alpha=0.5)

min_v = min(pred_df["actual"].min(), pred_df["predicted"].min())
max_v = max(pred_df["actual"].max(), pred_df["predicted"].max())

plt.plot([min_v, max_v], [min_v, max_v], "--")

plt.xlabel("Observed")
plt.ylabel("Predicted")
plt.title("GAT Spatial CV")

plt.show()